In [1]:
!pip install -q spacy pandas openpyxl

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import pandas as pd
import spacy
import os
import re
from spacy.pipeline import EntityRuler

In [4]:
BASE = "/content/drive/MyDrive/KG Team /Harini's Work Automation"

REFERENCE = f"{BASE}/Reference Data"
INPUTS = f"{BASE}/Input Texts"
VALIDATION = f"{BASE}/Validation Data"
OUTPUT = f"{BASE}/Output"

os.makedirs(OUTPUT, exist_ok=True)

print("Paths Loaded Successfully")

Paths Loaded Successfully


In [5]:
entity_df = pd.read_excel(
    f"{REFERENCE}/Entity_Master_List_Final.xlsx"
)

print("Total Entities:", len(entity_df))
entity_df.head()

Total Entities: 532


,Entity,Label,Alias
0,diabetes,DISEASE,DM; diabetic disorder; diabetic disorder
1,glucose,CHEMICAL,blood sugar
2,triglycerides,CHEMICAL,NaN
3,Ashwagandha,HERB,Indian ginseng; Withania somnifera; Withania s...
4,Withania somnifera,HERB,NaN


In [6]:
def normalize(text):

    text = str(text).lower().strip()

    text = text.replace("-", " ")
    text = text.replace("_", " ")

    text = re.sub(r"\s+", " ", text)

    return text

In [7]:
patterns = []

for _, row in entity_df.iterrows():

    entity = str(row["Entity"]).strip()
    label = str(row["Label"]).strip()

    patterns.append({
        "label": label,
        "pattern": entity
    })

    if pd.notna(row["Alias"]):

        aliases = str(row["Alias"]).split(";")

        for alias in aliases:

            alias = alias.strip()

            if alias:

                patterns.append({
                    "label": label,
                    "pattern": alias
                })

print("Total Patterns:", len(patterns))

Total Patterns: 845


In [8]:
nlp = spacy.blank("en")

ruler = nlp.add_pipe("entity_ruler")

ruler.add_patterns(patterns)

print("EntityRuler Loaded")

EntityRuler Loaded


In [9]:
def load_semantic_tags(filepath):

    tag_df = pd.read_excel(filepath)

    semantic_dict = {}

    for _, row in tag_df.iterrows():

        term = str(row["Term"]).strip().lower()

        semantic_dict[term] = {

            "Category": row.get("Category",""),

            "Tag1": row.get("Tag1",""),
            "Tag2": row.get("Tag2",""),
            "Tag3": row.get("Tag3",""),
            "Tag4": row.get("Tag4",""),
            "Tag5": row.get("Tag5","")
        }

    return semantic_dict

In [10]:
def annotate_document(text_file, semantic_file):

    semantic_dict = load_semantic_tags(
        semantic_file
    )

    with open(
        text_file,
        "r",
        encoding="utf-8"
    ) as f:

        text = f.read()

    doc = nlp(text)

    rows = []

    for ent in doc.ents:

        entity = ent.text.strip()

        key = entity.lower()

        tags = semantic_dict.get(
            key,
            {}
        )

        rows.append({

            "Entity": entity,
            "Label": ent.label_,

            "Category": tags.get("Category",""),

            "Tag1": tags.get("Tag1",""),
            "Tag2": tags.get("Tag2",""),
            "Tag3": tags.get("Tag3",""),
            "Tag4": tags.get("Tag4",""),
            "Tag5": tags.get("Tag5","")
        })

    return pd.DataFrame(rows)

In [11]:
def evaluate(auto_df, manual_file):

    manual = pd.read_excel(
        manual_file
    )

    manual_entities = set(

        manual["Entity"]
        .astype(str)
        .apply(normalize)
    )

    auto_entities = set(

        auto_df["Entity"]
        .astype(str)
        .apply(normalize)
    )

    matched = manual_entities.intersection(
        auto_entities
    )

    missing = manual_entities - auto_entities

    extra = auto_entities - manual_entities

    precision = (
        len(matched) /
        len(auto_entities) * 100
        if len(auto_entities) > 0 else 0
    )

    recall = (
        len(matched) /
        len(manual_entities) * 100
        if len(manual_entities) > 0 else 0
    )

    f1 = (
        2 * precision * recall /
        (precision + recall)
        if (precision + recall) > 0 else 0
    )

    return (
        matched,
        missing,
        extra,
        precision,
        recall,
        f1
    )

In [12]:
datasets = [

{
"name":"Ashwagandha",

"text":
f"{INPUTS}/cleaned_text_v5.txt",

"semantic":
f"{REFERENCE}/Ashwagandha Semantic_Tags.xlsx",

"manual":
f"{VALIDATION}/Ashwagandha Annotated_Entities.xlsx"
},

{
"name":"Clinical",

"text":
f"{INPUTS}/cleaned_text_v5-2.txt",

"semantic":
f"{REFERENCE}/Clinical_Evaluation Semantic_Tags.xlsx",

"manual":
f"{VALIDATION}/Clinical_Evaluation Annotated_Entities.xlsx"
},

{
"name":"Diabetes",

"text":
f"{INPUTS}/cleaned_text_v5-4.txt",

"semantic":
f"{REFERENCE}/Ayurveda_Management_of_Diabetes Semantic_Tag.xlsx",

"manual":
f"{VALIDATION}/Ayurveda_Management_of_Diabetes Annotated_Entities.xlsx"
}

]

In [13]:
metrics = []

for ds in datasets:

    print("\nProcessing:", ds["name"])

    auto_df = annotate_document(
        ds["text"],
        ds["semantic"]
    )

    auto_df.to_excel(
        f"{OUTPUT}/{ds['name']}_Auto_Annotations.xlsx",
        index=False
    )

    matched, missing, extra, precision, recall, f1 = evaluate(
        auto_df,
        ds["manual"]
    )

    pd.DataFrame(
        {"Matched": list(matched)}
    ).to_excel(
        f"{OUTPUT}/{ds['name']}_Matched.xlsx",
        index=False
    )

    pd.DataFrame(
        {"Missing": list(missing)}
    ).to_excel(
        f"{OUTPUT}/{ds['name']}_Missing.xlsx",
        index=False
    )

    pd.DataFrame(
        {"Extra": list(extra)}
    ).to_excel(
        f"{OUTPUT}/{ds['name']}_Extra.xlsx",
        index=False
    )

    metrics.append({

        "Dataset": ds["name"],

        "Matched": len(matched),
        "Missing": len(missing),
        "Extra": len(extra),

        "Precision": round(precision,2),
        "Recall": round(recall,2),
        "F1": round(f1,2)
    })

    print(
        round(precision,2),
        round(recall,2),
        round(f1,2)
    )


Processing: Ashwagandha
82.65 83.08 82.86

Processing: Clinical
47.62 57.14 51.95

Processing: Diabetes
80.56 92.55 86.14


In [14]:
metrics_df = pd.DataFrame(metrics)

metrics_df.to_excel(
    f"{OUTPUT}/Final_Metrics.xlsx",
    index=False
)

metrics_df

,Dataset,Matched,Missing,Extra,Precision,Recall,F1
0,Ashwagandha,324,66,68,82.65,83.08,82.86
1,Clinical,20,15,22,47.62,57.14,51.95
2,Diabetes,87,7,21,80.56,92.55,86.14
